In [ ]:
import os
import sys
import time
import random
import csv
from pathlib import Path
from dataclasses import dataclass
from functools import lru_cache

import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.filters import apply_hysteresis_threshold, sato
from skimage.measure import regionprops, label as label_components
from skimage.morphology import skeletonize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tqdm.auto import tqdm
import kagglehub

In [ ]:
@dataclass
class Config:
    #Preprocessing & Denoising
    median_k: int = 5
    bilateral_d: int = 7
    bilateral_sigma_color: float = 40.0
    bilateral_sigma_space: float = 40.0

    #Restoration & Frequency Filters
    use_homomorphic: bool = True
    homo_gamma_l: float = 0.5
    homo_gamma_h: float = 1.5
    homo_c: float = 1.0
    homo_d0: float = 30.0
    butter_d0: float = 60.0
    butter_n: int = 2
    bg_k: int = 31

    #Contrast Enhancement & Sharpening
    use_clahe: bool = False
    clahe_clip: float = 2.0
    clahe_grid: int = 8
    sharpen_mode: str = "highboost"  # Options: "highboost", "none"
    sharpen_sigma: float = 1.5
    sharpen_k: float = 1.0

    #Feature Extraction & Multi-Scale Ridge Fusion
    sobel_t: float = 40.0
    canny_lo: int = 15
    canny_hi: int = 45
    lap_sigma: float = 1.0
    min_contrast: float = 10.0

    use_ridge_fusion: bool = True
    ridge_sigmas: tuple = (1.0, 2.0, 3.0)
    blackhat_k: int = 15
    gabor_ksize: int = 21
    gabor_sigma: float = 4.0
    gabor_lambda: float = 10.0
    gabor_orients: int = 8
    hyst_lo: float = 0.25
    hyst_hi: float = 0.50

    #Morphological Cleaning & Component Filtering
    link_len: int = 9
    min_len: int = 20           # Minimum skeleton pixels per crack segment
    sure_len: int = 60          # Long components skip shape filtering
    min_elong: float = 3.0      # Major/minor axis ratio (rejects round stains/pores)
    detect_len: int = 25        # Skeleton pixels required to flag image as cracked

    #Calibration & Output Settings
    mm_per_px: float = 1.0      # Physical calibration scale factor (1.0 = pixels)
    is_calibrated: bool = False # Set True only when mm_per_px is calibrated
    severity_bins: tuple = (0.3, 1.0) # Bins: (hairline < 0.3mm, moderate < 1.0mm, severe >= 1.0mm)

    random_seed: int = 42
    output_dir: str = "crackvision_outputs"
    save_outputs: bool = True

CFG = Config()
random.seed(CFG.random_seed)
np.random.seed(CFG.random_seed)

print("Configuration initialized successfully!")

Configuration initialized successfully!


In [ ]:
# =============================================================================
# Cell 3: Image I/O & Low-Level Image Operators
# =============================================================================
def load_image(image_path):
    """Safely load an image from disk."""
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Path does not exist: {path.resolve()}")
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not decode image at: {path.resolve()}")
    return bgr

def normalize_u8(image):
    """Safely convert a float map to displayable uint8 [0, 255]."""
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=255.0, neginf=0.0)
    return np.clip(image, 0, 255).astype(np.uint8)

@lru_cache(maxsize=16)
def distance_grid(shape):
    """Frequency-domain distance grid centered with np.fft.fftshift."""
    h, w = shape
    y = np.arange(h)[:, None] - h // 2
    x = np.arange(w)[None, :] - w // 2
    return np.hypot(x, y)

@lru_cache(maxsize=16)
def butterworth_lowpass_kernel(shape, d0, n):
    """Generate Butterworth low-pass filter mask."""
    d = distance_grid(shape)
    d0 = max(float(d0), 1e-6)
    return (1.0 / (1.0 + (d / d0) ** (2 * n))).astype(np.float32)

@lru_cache(maxsize=16)
def homomorphic_kernel(shape, gl, gh, c, d0):
    """Generate Homomorphic filter mask for shadow & illumination correction."""
    d = distance_grid(shape)
    d0 = max(float(d0), 1e-6)
    return ((gh - gl) * (1.0 - np.exp(-c * (d ** 2) / (d0 ** 2))) + gl).astype(np.float32)

def apply_frequency_filter(img, H):
    """Apply a 2D frequency domain filter mask H using FFT."""
    img_f = img.astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(img_f))
    res = np.fft.ifft2(np.fft.ifftshift(F * H))
    return np.real(res).astype(np.float32)

def homomorphic_correct(gray, cfg=CFG):
    """Correct non-uniform illumination and shadows via Homomorphic filtering."""
    H = homomorphic_kernel(gray.shape, cfg.homo_gamma_l, cfg.homo_gamma_h, cfg.homo_c, cfg.homo_d0)
    z = apply_frequency_filter(np.log1p(np.maximum(gray.astype(np.float32), 0)), H)
    out = np.expm1(z)
    mean_scale = float(gray.mean()) / max(float(out.mean()), 1e-6)
    out *= mean_scale
    return normalize_u8(out)

def high_boost_filter(img, sigma=1.5, k=1.0):
    """Spatial High-Boost sharpening."""
    img_f = img.astype(np.float32)
    blurred = cv2.GaussianBlur(img_f, (0, 0), sigma)
    mask = img_f - blurred
    return np.clip(img_f + k * mask, 0, 255).astype(np.float32)

print("Utility functions ready!")

Utility functions ready!


In [ ]:
# =============================================================================
# Cell 4A: Preprocessing, Feature Extraction & Ridge Fusion (Stages 1 - 3)
# =============================================================================

# --- Stage 1 & 2: Preprocessing & Restoration ---
def preprocess_image(gray, cfg=CFG):
    median = cv2.medianBlur(gray, cfg.median_k if cfg.median_k % 2 != 0 else cfg.median_k + 1)
    bilateral = cv2.bilateralFilter(median, cfg.bilateral_d, cfg.bilateral_sigma_color, cfg.bilateral_sigma_space)

    restored = homomorphic_correct(bilateral, cfg) if cfg.use_homomorphic else bilateral.copy()

    if cfg.use_clahe:
        clahe = cv2.createCLAHE(clipLimit=cfg.clahe_clip, tileGridSize=(cfg.clahe_grid, cfg.clahe_grid))
        enhanced = clahe.apply(restored)
    else:
        enhanced = restored.copy()

    H_butter = butterworth_lowpass_kernel(enhanced.shape, cfg.butter_d0, cfg.butter_n)
    smooth = apply_frequency_filter(enhanced, H_butter)
    bg_k = cfg.bg_k if cfg.bg_k % 2 != 0 else cfg.bg_k + 1
    background = cv2.medianBlur(enhanced, bg_k).astype(np.float32)
    bg_sub = np.clip(background - smooth, 0, 255)

    sharpened = high_boost_filter(bg_sub, cfg.sharpen_sigma, cfg.sharpen_k) if cfg.sharpen_mode == "highboost" else bg_sub

    return {
        "median": median, "bilateral": bilateral, "restored": restored,
        "enhanced": enhanced, "smooth": smooth, "background": background,
        "bg_sub": bg_sub, "sharpened": sharpened
    }

# --- Stage 3: Feature Extraction & Ridge Fusion ---
def gabor_response(img, cfg=CFG):
    img_f = img.astype(np.float32)
    resp = np.zeros_like(img_f)
    for theta in np.linspace(0, np.pi, cfg.gabor_orients, endpoint=False):
        k = cv2.getGaborKernel((cfg.gabor_ksize, cfg.gabor_ksize), cfg.gabor_sigma, theta, cfg.gabor_lambda, 0.5, 0, ktype=cv2.CV_32F)
        k -= k.mean()
        filtered = cv2.filter2D(img_f, cv2.CV_32F, k)
        resp = np.maximum(resp, filtered)
    return resp

def ridge_score_map(pre_img, enh_img, cfg=CFG):
    sato_r = sato(enh_img, sigmas=cfg.ridge_sigmas, black_ridges=False)
    gabor_r = gabor_response(enh_img, cfg)
    ksize = cfg.blackhat_k if cfg.blackhat_k % 2 != 0 else cfg.blackhat_k + 1
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    blackhat = cv2.morphologyEx(pre_img, cv2.MORPH_BLACKHAT, se).astype(np.float32)

    def norm01(x):
        hi = np.percentile(x, 99.5)
        return np.clip(x / hi, 0, 1).astype(np.float32) if hi > 0 else np.zeros_like(x, np.float32)

    return (norm01(sato_r) + norm01(gabor_r) + norm01(blackhat)) / 3.0

def extract_features(prep_dict, cfg=CFG):
    enh = prep_dict["sharpened"]
    pre = prep_dict["enhanced"]
    smooth = prep_dict["smooth"]
    enh8 = normalize_u8(enh)

    sobel_x = cv2.Sobel(enh, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(enh, cv2.CV_32F, 0, 1, ksize=3)
    sobel = np.hypot(sobel_x, sobel_y)
    canny = cv2.Canny(enh8, cfg.canny_lo, cfg.canny_hi)
    laplacian = cv2.Laplacian(cv2.GaussianBlur(smooth, (0, 0), cfg.lap_sigma), cv2.CV_32F)

    otsu_t, _ = cv2.threshold(enh8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    t = max(float(otsu_t), cfg.min_contrast)
    edges = ((canny > 0) | (sobel > cfg.sobel_t)).astype(np.uint8)
    near_edge = cv2.dilate(edges, np.ones((5, 5), np.uint8)).astype(bool)
    candidate = (enh > t) & (laplacian > 0) & near_edge

    ridge_score = ridge_score_map(pre, enh, cfg)
    ridge_mask = apply_hysteresis_threshold(ridge_score, cfg.hyst_lo, cfg.hyst_hi)
    final_candidate = candidate | (ridge_mask & (enh > cfg.min_contrast))

    return {
        "sobel": sobel, "canny": canny, "laplacian": laplacian,
        "ridge_score": ridge_score, "ridge_mask": ridge_mask,
        "candidate": final_candidate
    }

print("Stages 1-3 functions loaded!")

Stages 1-3 functions loaded!


Pratho Starts here

In [ ]:
# =============================================================================
# Cell 4B: Mask Refinement, Skeletonization & Measurements (Stages 4 - 6)
# =============================================================================

# --- Stage 4 & 5: Mask Refinement & Skeletonization ---
@lru_cache(maxsize=32)
def _line_kernel(length, angle):
    kernel = np.zeros((length, length), np.uint8)
    c = length // 2
    rad = np.deg2rad(angle)
    dx, dy = np.cos(rad) * c, np.sin(rad) * c
    cv2.line(kernel, (round(c - dx), round(c - dy)), (round(c + dx), round(c + dy)), 1, 1)
    return kernel

def refine_crack_mask(binary_candidate, cfg=CFG):
    linked = np.zeros_like(binary_candidate, dtype=np.uint8)
    cand_u8 = binary_candidate.astype(np.uint8)
    for ang in (0, 45, 90, 135):
        linked |= cv2.morphologyEx(cand_u8, cv2.MORPH_CLOSE, _line_kernel(cfg.link_len, ang))

    labels = label_components(linked, connectivity=2)
    skel_temp = skeletonize(linked.astype(bool), method="zhang")

    n_labels = labels.max() + 1
    skel_lengths = np.bincount(labels[skel_temp], minlength=n_labels)
    keep_lookup = skel_lengths >= cfg.min_len

    for prop in regionprops(labels):
        lbl = prop.label
        if not keep_lookup[lbl] or skel_lengths[lbl] >= cfg.sure_len:
            continue
        elongation = prop.axis_major_length / max(prop.axis_minor_length, 1.0)
        if elongation < cfg.min_elong:
            keep_lookup[lbl] = False

    keep_lookup[0] = False
    refined_mask = keep_lookup[labels].astype(np.uint8)
    return linked, refined_mask

def compute_skeleton(refined_mask):
    return skeletonize(refined_mask.astype(bool), method="zhang").astype(bool)

# --- Stage 6: Crack Quantification ---
def measure_cracks(refined_mask, skeleton, cfg=CFG):
    dist_transform = cv2.distanceTransform(refined_mask.astype(np.uint8), cv2.DIST_L2, 3)
    widths_px = 2.0 * dist_transform[skeleton]

    length_px = int(skeleton.sum())
    mean_width_px = float(widths_px.mean()) if widths_px.size > 0 else 0.0
    max_width_px = float(widths_px.max()) if widths_px.size > 0 else 0.0

    length_val = length_px * cfg.mm_per_px
    mean_width_val = mean_width_px * cfg.mm_per_px
    max_width_val = max_width_px * cfg.mm_per_px

    ys, xs = np.nonzero(skeleton)
    if xs.size >= 2:
        cov = np.cov(np.vstack([xs, ys]))
        _, vecs = np.linalg.eigh(cov)
        vx, vy = vecs[:, -1]
        angle_deg = float(np.degrees(np.arctan2(-vy, vx)) % 180)
    else:
        angle_deg = float("nan")

    if np.isnan(angle_deg): orientation = "None"
    elif angle_deg < 22.5 or angle_deg >= 157.5: orientation = "Horizontal"
    elif 67.5 <= angle_deg < 112.5: orientation = "Vertical"
    else: orientation = "Diagonal"

    if max_width_val <= 0: severity = "None"
    elif max_width_val < cfg.severity_bins[0]: severity = "Hairline"
    elif max_width_val < cfg.severity_bins[1]: severity = "Moderate"
    else: severity = "Severe"

    num_components = int(label_components(refined_mask, connectivity=2).max())
    crack_percentage = float((refined_mask.sum() / refined_mask.size) * 100.0)
    is_cracked = bool(length_px >= cfg.detect_len)

    return {
        "length_px": length_px, "mean_width_px": mean_width_px, "max_width_px": max_width_px,
        "length_val": length_val, "mean_width_val": mean_width_val, "max_width_val": max_width_val,
        "num_components": num_components, "crack_percentage": crack_percentage,
        "orientation": orientation, "angle_deg": angle_deg, "severity": severity,
        "is_cracked": is_cracked, "dist_transform": dist_transform
    }

print("Stages 4-6 functions loaded!")

Stages 4-6 functions loaded!


In [ ]:
# =============================================================================
# Cell 5: Master Detector Integration
# =============================================================================
def detect_cracks(bgr_image, cfg=CFG):
    """Run full 6-stage CrackVision pipeline on a BGR image."""
    gray = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2GRAY)

    prep_dict = preprocess_image(gray, cfg)
    feat_dict = extract_features(prep_dict, cfg)
    linked_mask, refined_mask = refine_crack_mask(feat_dict["candidate"], cfg)
    skeleton = compute_skeleton(refined_mask)
    measures = measure_cracks(refined_mask, skeleton, cfg)

    # Generate Overlay
    overlay = bgr_image.copy()
    overlay[refined_mask.astype(bool)] = (
        0.5 * overlay[refined_mask.astype(bool)] + 0.5 * np.array([0, 255, 0])
    ).astype(np.uint8)
    skel_dilated = cv2.dilate(skeleton.astype(np.uint8), np.ones((2, 2), np.uint8)).astype(bool)
    overlay[skel_dilated] = (0, 0, 255)  # Red skeleton line

    return {
        "gray": gray, **prep_dict, **feat_dict,
        "linked_mask": linked_mask, "refined_mask": refined_mask,
        "skeleton": skeleton, "overlay": overlay, **measures
    }

print("Master detector loaded!")

Master detector loaded!


In [ ]:
# =============================================================================
# Cell 6: Visualization & Reporting Utilities
# =============================================================================
def display_results(bgr_image, results, title_prefix="CrackVision Analysis", cfg=CFG):
    """Renders a clean 13-stage image subplot grid."""
    unit_str = "mm" if cfg.is_calibrated else "px"

    visual_stages = [
        ("1. Original RGB Image", cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)),
        ("2. Grayscale Input", results["gray"]),
        ("3. Preprocessed & Denoised", results["enhanced"]),
        ("4. Background Subtraction", results["bg_sub"]),
        ("5. Sobel Gradient Magnitude", normalize_u8(results["sobel"])),
        ("6. Canny Edge Detector", results["canny"]),
        ("7. Laplacian Edge Map", normalize_u8(np.abs(results["laplacian"]))),
        ("8. Fused Ridge Score Map", normalize_u8(results["ridge_score"] * 255)),
        ("9. Candidate Binary Mask", results["candidate"].astype(np.uint8) * 255),
        ("10. Final Refined Crack Mask", results["refined_mask"] * 255),
        ("11. Zhang-Suen Skeleton", results["skeleton"].astype(np.uint8) * 255),
        ("12. Distance Transform Map", normalize_u8(cv2.normalize(results["dist_transform"], None, 0, 255, cv2.NORM_MINMAX))),
        ("13. Detection Overlay (Red=Skel)", cv2.cvtColor(results["overlay"], cv2.COLOR_BGR2RGB))
    ]

    fig, axes = plt.subplots(4, 4, figsize=(18, 14))
    axes = axes.ravel()

    for idx, (title, img) in enumerate(visual_stages):
        if img.ndim == 3:
            axes[idx].imshow(img)
        else:
            axes[idx].imshow(img, cmap="gray" if "Distance" not in title else "jet")
        axes[idx].set_title(title, fontsize=10, fontweight="bold")
        axes[idx].axis("off")

    for idx in range(len(visual_stages), len(axes)):
        axes[idx].axis("off")

    fig.suptitle(
        f"{title_prefix} | Detected: {'CRACK' if results['is_cracked'] else 'NO CRACK'} | "
        f"Length: {results['length_val']:.1f}{unit_str} | Max Width: {results['max_width_val']:.2f}{unit_str}",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

def print_console_summary(image_name, results, cfg=CFG):
    """Outputs text summary to the console."""
    unit = "mm" if cfg.is_calibrated else "px"
    print("\n" + "=" * 60)
    print(f"   CRACKVISION ANALYSIS SUMMARY: {image_name}")
    print("=" * 60)
    print(f"Status               : {'CRACK DETECTED' if results['is_cracked'] else 'NO CRACK DETECTED'}")
    print(f"Total Skeleton Length: {results['length_val']:.2f} {unit} ({results['length_px']} pixels)")
    print(f"Average Crack Width  : {results['mean_width_val']:.2f} {unit} ({results['mean_width_px']:.2f} pixels)")
    print(f"Maximum Crack Width  : {results['max_width_val']:.2f} {unit} ({results['max_width_px']:.2f} pixels)")
    print(f"Connected Components : {results['num_components']}")
    print(f"Crack Pixel Density  : {results['crack_percentage']:.3f}% of total area")
    print(f"Dominant Orientation : {results['orientation']} ({results['angle_deg']:.1f}°)")
    print(f"Severity Bin         : {results['severity']}")
    print("=" * 60 + "\n")

print("Visualization functions loaded!")

Visualization functions loaded!


In [ ]:
# =============================================================================
# Cell 7: Evaluation Metrics Modules
# =============================================================================
def evaluate_pixel_segmentation(pred_mask, gt_mask):
    """Compute pixel-level segmentation metrics against a ground-truth binary mask."""
    pred_bin = (pred_mask > 0).astype(np.uint8).ravel()
    gt_bin = (gt_mask > 0).astype(np.uint8).ravel()

    if pred_bin.shape != gt_bin.shape:
        raise ValueError(f"Shape mismatch: pred_mask {pred_bin.shape} vs gt_mask {gt_bin.shape}")

    tn, fp, fn, tp = confusion_matrix(gt_bin, pred_bin, labels=[0, 1]).ravel()

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1e-9)
    rec = tp / max(tp + fn, 1e-9)
    spec = tn / max(tn + fp, 1e-9)
    f1 = 2 * (prec * rec) / max(prec + rec, 1e-9)
    iou = tp / max(tp + fp + fn, 1e-9)

    return {
        "Accuracy": acc * 100.0, "Precision": prec * 100.0, "Recall": rec * 100.0,
        "Specificity": spec * 100.0, "F1-Score": f1 * 100.0, "IoU": iou * 100.0,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

def display_evaluation_report(metrics_dict):
    """Print formatted evaluation report."""
    print("\n" + "=" * 60)
    print("      PIXEL-LEVEL SEGMENTATION EVALUATION REPORT")
    print("=" * 60)
    print(f" Accuracy    : {metrics_dict['Accuracy']:.2f}%")
    print(f" Precision   : {metrics_dict['Precision']:.2f}%")
    print(f" Recall (Sens): {metrics_dict['Recall']:.2f}%")
    print(f" Specificity : {metrics_dict['Specificity']:.2f}%")
    print(f" F1-Score    : {metrics_dict['F1-Score']:.2f}%")
    print(f" IoU Score   : {metrics_dict['IoU']:.2f}%")
    print("-" * 60)
    print(" Confusion Matrix:")
    print(f"   [ Actual BG  (0) ] -> TN: {metrics_dict['TN']:<8} | FP: {metrics_dict['FP']:<8}")
    print(f"   [ Actual Crack(1) ] -> FN: {metrics_dict['FN']:<8} | TP: {metrics_dict['TP']:<8}")
    print("=" * 60 + "\n")

def evaluate_dataset_classification(positive_paths, negative_paths, max_images=100, cfg=CFG):
    """Evaluate image classification statistics on Kaggle Positive/Negative splits."""
    samples = [(p, 1) for p in positive_paths] + [(p, 0) for p in negative_paths]
    random.shuffle(samples)
    if max_images is not None:
        samples = samples[:max_images]

    y_true, y_pred, records = [], [], []
    t0 = time.time()

    for path, label in tqdm(samples, desc="Benchmarking Dataset"):
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None: continue

        res = detect_cracks(bgr, cfg)
        pred = int(res["is_cracked"])

        y_true.append(label)
        y_pred.append(pred)
        records.append({
            "path": str(path), "true_label": label, "predicted_label": pred,
            "length_px": res["length_px"], "max_width_px": res["max_width_px"],
            "severity": res["severity"]
        })

    elapsed = time.time() - t0
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    print("\n" + "=" * 60)
    print("   FOLDER-LEVEL IMAGE CLASSIFICATION BENCHMARK")
    print("=" * 60)
    print(f" Images Evaluated : {len(y_true)} ({elapsed:.1f}s total | {1000 * elapsed / len(y_true):.0f} ms/img)")
    print(f" Accuracy         : {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f" Precision        : {precision_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print(f" Recall           : {recall_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print(f" F1-Score         : {f1_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print(f" Confusion Matrix : [TN={tn}, FP={fp}, FN={fn}, TP={tp}]")
    print("=" * 60 + "\n")

    if cfg.save_outputs:
        out_dir = Path(cfg.output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        csv_path = out_dir / "classification_benchmark.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
            writer.writeheader()
            writer.writerows(records)
        print(f"Saved dataset classification log to: {csv_path.resolve()}")

print("Evaluation engine ready!")

Evaluation engine ready!


In [ ]:
# =============================================================================
# Cell A: Dataset Ingestion & Discovery
# =============================================================================
print("Step 1: Ingesting Dataset...")

try:
    root_path = Path(kagglehub.dataset_download("arunrk7/surface-crack-detection"))
    positive_images = sorted(root_path.rglob("Positive/*.jpg"))
    negative_images = sorted(root_path.rglob("Negative/*.jpg"))

    print(f"\nDataset Root Directory : {root_path.resolve()}")
    print(f"Total Positive Images  : {len(positive_images):,}")
    print(f"Total Negative Images  : {len(negative_images):,}")

    if not positive_images or not negative_images:
        raise FileNotFoundError("Could not locate Positive or Negative image subfolders.")

except Exception as e:
    print(f"KaggleHub download skipped or failed: {e}")
    positive_images, negative_images = [], []

print("\nDataset ready for testing!")

Step 1: Ingesting Dataset...
Using Colab cache for faster access to the 'surface-crack-detection' dataset.

Dataset Root Directory : /kaggle/input/surface-crack-detection
Total Positive Images  : 20,000
Total Negative Images  : 20,000

Dataset ready for testing!


In [ ]:
SAMPLE_LIMIT_PER_CLASS = 1000

if positive_images and negative_images:
    pos_samples = [(p, 1) for p in random.sample(positive_images, min(SAMPLE_LIMIT_PER_CLASS, len(positive_images)))]
    neg_samples = [(p, 0) for p in random.sample(negative_images, min(SAMPLE_LIMIT_PER_CLASS, len(negative_images)))]

    test_queue = pos_samples + neg_samples
    random.shuffle(test_queue)  # Shuffle order for unbiased execution

    test_results_log = []

    for img_path, true_label in tqdm(test_queue, desc="Testing Files"):
        bgr_img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if bgr_img is None:
            continue

        # Run detection pipeline
        res = detect_cracks(bgr_img, CFG)
        pred_label = int(res["is_cracked"])

        # Record file-by-file log
        test_results_log.append({
            "path": img_path,
            "filename": img_path.name,
            "true_label": true_label,            # 1 = Positive (Cracked), 0 = Negative (Uncracked)
            "predicted_label": pred_label,      # 1 = Positive (Cracked), 0 = Negative (Uncracked)
            "is_correct": (true_label == pred_label),
            "length_px": res["length_px"],
            "max_width_px": res["max_width_px"],
            "severity": res["severity"]
        })

    print(f"\nCompleted testing on {len(test_results_log)} images!")
else:
    print("No images loaded. Please execute Cell A first.")

Testing Files:   0%|          | 0/2000 [00:00<?, ?it/s]


Completed testing on 2000 images!


In [ ]:
# =============================================================================
# Cell C: Classification Evaluation Report
# =============================================================================
print("Step 3: Generating Classification Evaluation Report...\n")

if 'test_results_log' in locals() and test_results_log:
    y_true = [item["true_label"] for item in test_results_log]
    y_pred = [item["predicted_label"] for item in test_results_log]

    # Calculate Confusion Matrix: TN, FP, FN, TP
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # Calculate Class-Specific Scores
    total_positive = tp + fn
    total_negative = tn + fp

    pos_accuracy = (tp / total_positive * 100.0) if total_positive > 0 else 0.0
    neg_accuracy = (tn / total_negative * 100.0) if total_negative > 0 else 0.0
    overall_accuracy = ((tp + tn) / len(y_true) * 100.0) if len(y_true) > 0 else 0.0

    print("=" * 65)
    print("        CRACKVISION CLASSIFICATION EVALUATION REPORT")
    print("=" * 65)
    print(f" Total Images Evaluated     : {len(y_true)}")
    print(f" Overall Accuracy           : {overall_accuracy:.2f}%\n")

    print("-" * 65)
    print(" 1. POSITIVE IMAGES SCORE (Ground Truth: Cracked)")
    print("-" * 65)
    print(f"   * Total Positive Images  : {total_positive}")
    print(f"   * Correctly Classified   : {tp} (True Positives)")
    print(f"   * Incorrectly Classified : {fn} (False Negatives / Missed Cracks)")
    print(f"   * Positive Class Score   : {pos_accuracy:.2f}% Correct\n")

    print("-" * 65)
    print(" 2. NEGATIVE IMAGES SCORE (Ground Truth: Non-Cracked)")
    print("-" * 65)
    print(f"   * Total Negative Images  : {total_negative}")
    print(f"   * Correctly Classified   : {tn} (True Negatives)")
    print(f"   * Incorrectly Classified : {fp} (False Positives / False Alarms)")
    print(f"   * Negative Class Score   : {neg_accuracy:.2f}% Correct\n")

    print("=" * 65)
    print(" SUMMARY METRICS")
    print("=" * 65)
    print(f" Precision (Exactness)    : {precision_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print(f" Recall / Sensitivity     : {recall_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print(f" F1-Score (Balance)       : {f1_score(y_true, y_pred, zero_division=0) * 100:.2f}%")
    print("=" * 65)

    # Display Misclassified File Details for Error Inspection
    # misclassified = [item for item in test_results_log if not item["is_correct"]]
    # if misclassified:
    #     print(f"\nMisclassified Files Inspection ({len(misclassified)} files):")
    #     for item in misclassified:
    #         gt_str = "Positive (Cracked)" if item["true_label"] == 1 else "Negative (Non-Cracked)"
    #         pred_str = "Positive (Cracked)" if item["predicted_label"] == 1 else "Negative (Non-Cracked)"
            #print(f" - {item['filename']:<20} | True: {gt_str:<20} | Pred: {pred_str}")

else:
    print("No test results found. Please run Cell B first.")

Step 3: Generating Classification Evaluation Report...

        CRACKVISION CLASSIFICATION EVALUATION REPORT
 Total Images Evaluated     : 2000
 Overall Accuracy           : 82.15%

-----------------------------------------------------------------
 1. POSITIVE IMAGES SCORE (Ground Truth: Cracked)
-----------------------------------------------------------------
   * Total Positive Images  : 1000
   * Correctly Classified   : 1000 (True Positives)
   * Incorrectly Classified : 0 (False Negatives / Missed Cracks)
   * Positive Class Score   : 100.00% Correct

-----------------------------------------------------------------
 2. NEGATIVE IMAGES SCORE (Ground Truth: Non-Cracked)
-----------------------------------------------------------------
   * Total Negative Images  : 1000
   * Correctly Classified   : 643 (True Negatives)
   * Incorrectly Classified : 357 (False Positives / False Alarms)
   * Negative Class Score   : 64.30% Correct

 SUMMARY METRICS
 Precision (Exactness)    : 73.69%